In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.font_manager import FontProperties

In [2]:
# -------- config --------
SHOW_TEXT = True  # set to False for no text

# Choose color scheme
COLOR_MODE = "white"   # "duke" or "white"

if COLOR_MODE == "white":
    PRIMARY = "#FFFFFF"
    ACCENT  = "#FFFFFF"
    TEXT    = "#FFFFFF"
else:
    PRIMARY = "#012169"  # Duke navy
    ACCENT  = "#012169"
    TEXT    = PRIMARY

TEXT = PRIMARY
FONTFACE = "Gill Sans"

OUTPUT_DIR = "../output_data/logo_output"
PNG_NAME = "carolina_logo.png"
JPG_NAME = "carolina_logo.jpg"
SVG_NAME = "carolina_logo.svg"
DPI = 300
FIGSIZE = (8, 6)

TILT_ANGLE = -60  # degrees
LW = 4.5  # overall line width for arms and ellipse

# Ellipse (envelope) parameters
ELLIPSE_WIDTH = 4.8   # total width
ELLIPSE_HEIGHT = 2.8   # total height
ELLIPSE_ANGLE = TILT_ANGLE

# Arc (C-shape) parameters in the ellipse's local frame (degrees)
ARC_DEG = 20
ARC_START_DEG = -270 + ARC_DEG  # start angle
ARC_END_DEG = 90 - ARC_DEG  # end angle (opening on the right)
ARC_DASHES = (15, 6)  # dash pattern
ARC_LW = LW  # line width of the arc

# Spiral params (normalized, then scaled to ellipse)
THETA_MAX = 2 * np.pi    # total turns
ARM_PHASE = np.pi      # separation between arms
B_SHAPE = 2.0        # controls how fast radius grows (>=1)
FILL = 0.92       # fraction of ellipse radii to use (<1 keeps margin)
LINEWIDTH = LW

# Arrow
ARROW_LEN = 2
ARROW_LW = 2.5

TITLE = "CAROLINA"
SUBTITLE = "Connecting Analyses and Research \n On Lensing and INtrinsic Alignments"
TITLE_FS = 40
SUB_FS = 20
ITAL_SUB = True

# -------- helpers --------
def rot2d(theta_deg):
    t = np.deg2rad(theta_deg)
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s], [s, c]])

def normalized_spiral(theta, b=B_SHAPE):
    """
    Monotonic radius r in [0,1], independent of absolute scaling.
    r(theta) = ((1 + theta/theta_max)^b - 1) / ((1 + 1)^b - 1)
    - Starts at 0 and ends at 1 exactly (nice containment after scaling).
    - b controls steepness (b>1 more compact center, b<1 more open).
    """
    u = (1.0 + theta/THETA_MAX)
    r = (u**b - 1.0) / ((1.0 + 1.0)**b - 1.0)  # denominator = 2^b - 1
    return np.clip(r, 0.0, 1.0)

def pts_to_figfrac(pts, fig_height_in):
    # 72 points = 1 inch
    return pts / (72.0 * fig_height_in)

def draw_logo():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    fig = plt.figure(figsize=FIGSIZE)

    if SHOW_TEXT:
        # Reserve a dynamic band at the top for title + subtitle (+gap)
        text_band_pts = 1.1*TITLE_FS + 1.1*SUB_FS + 25  # scale + 10pt gap
        top_pad = pts_to_figfrac(text_band_pts, FIGSIZE[1])
        # axes: [left, bottom, width, height] in figure fraction
        axes_rect = [0.08, 0.08, 0.84, 1.0 - 0.08 - top_pad]
    else:
        axes_rect = [0.06, 0.06, 0.88, 0.88]

    ax = fig.add_axes(axes_rect)
    ax.set_aspect('equal')
    ax.axis('off')

    # Ellipse radii (semi-axes) in data coords
    a = ELLIPSE_WIDTH / 2.0
    b = ELLIPSE_HEIGHT / 2.0

    # Generate spiral in the ellipse's *local* (unrotated) frame,
    # then scale by a*FILL along x and b*FILL along y, then rotate.
    theta = np.linspace(0, THETA_MAX, 1200)
    r = normalized_spiral(theta, b=B_SHAPE)  # in [0,1]
    x = r * np.cos(theta)
    y = r * np.sin(theta)

    # Second arm (phase-shifted), also normalized
    r2 = normalized_spiral(theta, b=B_SHAPE)
    x2 = r2 * np.cos(theta + ARM_PHASE)
    y2 = r2 * np.sin(theta + ARM_PHASE)

    # Anisotropic scale to ellipse radii (shrink by FILL for margin)
    X  = np.vstack([a * FILL * x,  b * FILL * y])
    X2 = np.vstack([a * FILL * x2, b * FILL * y2])

    # Rotate both arms and ellipse together
    arm_rot = rot2d(TILT_ANGLE)
    arm1 = arm_rot @ X
    arm2 = arm_rot @ X2

    # Plot arms
    ax.plot(arm1[0], arm1[1], color=PRIMARY, lw=LINEWIDTH, solid_capstyle='round')
    ax.plot(arm2[0], arm2[1], color=PRIMARY,  lw=LINEWIDTH, solid_capstyle='round')

    # C-shaped dashed arc instead of full ellipse ---
    # sample parametric ellipse in its local (unrotated) frame
    tt = np.deg2rad(np.linspace(ARC_START_DEG, ARC_END_DEG, 400))
    x_e = (ELLIPSE_WIDTH / 2.0)  * np.cos(tt)
    y_e = (ELLIPSE_HEIGHT / 2.0) * np.sin(tt)

    # rotate arc to match ellipse tilt
    arc_rot = rot2d(TILT_ANGLE)
    arc = arc_rot @ np.vstack([x_e, y_e])

    # draw dashed arc (reads as a "C")
    ax.plot(arc[0], arc[1], color=PRIMARY, lw=ARC_LW, solid_capstyle='round')

    # Central bulge (simple filled circle) ---
    bulge_r = min(a, b) * 0.15  # ~15% of semi-minor axis; tweak to taste
    bulge = Circle((0, 0), radius=bulge_r, facecolor=PRIMARY, edgecolor='none', zorder=0)
    ax.add_patch(bulge)

    # --- Arrow tilted by the same angle as ellipse/spiral ---
    # base (unrotated) start and direction
    start_local = np.array([0.0, -0.0])  # where the arrow starts (before rotation)
    dir_local   = np.array([0.0, ARROW_LEN])  # arrow points "up" before rotation

    R = rot2d(ELLIPSE_ANGLE)  # same rotation used elsewhere
    start_rot = (R @ start_local).tolist()
    end_rot   = (R @ (start_local + dir_local)).tolist()

    arrow = FancyArrowPatch(
        start_rot, end_rot,
        arrowstyle='Simple,head_length=10,head_width=8,tail_width=3',
        linewidth=ARROW_LW, color=PRIMARY
    )
    ax.add_patch(arrow)

    # Define font properties
    title_fp = FontProperties(family=FONTFACE, weight='light')
    sub_fp   = FontProperties(family=FONTFACE, weight='light',
                              style='italic' if ITAL_SUB else 'normal')

    if SHOW_TEXT:
        band_bottom = axes_rect[1] + axes_rect[3]
        band_top    = 0.995  # small headroom at top edge

        # --- measure text heights in figure fraction ---
        fig_h_in = FIGSIZE[1]
        line_spacing = 1.15  # tweak if you want tighter/looser subtitle lines
        n_lines = len(SUBTITLE.split("\n"))

        title_h = (TITLE_FS / 72.0) / fig_h_in
        sub_h   = (n_lines * SUB_FS * line_spacing) / 72.0 / fig_h_in

        band_height = band_top - band_bottom

        # Equalize: (title-sub gap) == (subtitle-drawing gap) = G
        # band_height = title_h + sub_h + 2*G  ->  G = (band_height - title_h - sub_h)/2
        G = (band_height - title_h - sub_h) / 2.0
        if G < 0:
            # Not enough space: bump your reserved band (increase text_band_pts) or clamp
            G = 0.0

        # Positions
        title_y      = band_top
        subtitle_top = band_top - title_h - G
        sub_step     = (SUB_FS * line_spacing) / 72.0 / fig_h_in  # per-line step (top-anchored)

        # Draw title
        fig.text(0.5, title_y, TITLE, fontproperties=title_fp,
                 ha='center', va='top', fontsize=TITLE_FS, color=TEXT)

        # Draw subtitle lines (stacked downward)
        for i, line in enumerate(SUBTITLE.split("\n")):
            y = subtitle_top - i * sub_step
            fig.text(0.5, y, line, fontproperties=sub_fp,
                     ha='center', va='top', fontsize=SUB_FS, color=TEXT)

    # Frame
    margin = 1  # add small margin
    ax.set_xlim(-ELLIPSE_WIDTH/2 - margin, ELLIPSE_WIDTH/2 + margin)
    ax.set_ylim(-ELLIPSE_HEIGHT/2 - margin, ELLIPSE_HEIGHT/2 + margin)

    # Save
    suffix = "_text" if SHOW_TEXT else "_notext"
    c_suffix = "" if COLOR_MODE == "duke" else "_white"
    png_path = os.path.join(OUTPUT_DIR, PNG_NAME.replace(".png", f"{suffix}{c_suffix}.png"))
    jpg_path = os.path.join(OUTPUT_DIR, JPG_NAME.replace(".jpg", f"{suffix}{c_suffix}.jpg"))
    svg_path = os.path.join(OUTPUT_DIR, SVG_NAME.replace(".svg", f"{suffix}{c_suffix}.svg"))
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight', transparent=True)
    fig.savefig(jpg_path, dpi=DPI, bbox_inches='tight', transparent=True)
    fig.savefig(svg_path, dpi=DPI, bbox_inches='tight', transparent=True)

    plt.close(fig)

if __name__ == "__main__":
    for COLOR_MODE in ["duke", "white"]:
        if COLOR_MODE == "white":
            PRIMARY = ACCENT = TEXT = "#FFFFFF"
        else:
            PRIMARY = ACCENT = TEXT = "#012169"

        draw_logo()
        print(f"{COLOR_MODE} version saved to: {os.path.abspath(OUTPUT_DIR)}")


duke version saved to: /Users/niko/Documents/Research/CAROLINA/docs/output_data/logo_output
white version saved to: /Users/niko/Documents/Research/CAROLINA/docs/output_data/logo_output
